## Goal

Generate a synthetic CSV file that simulates collected rural community demand data.

In [17]:
import sys
from pathlib import Path

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from agents.generate_demo_data import create_demo_dataset

df = create_demo_dataset(rows=250)

df.head()


,response_id,age_group,gender,region,occupation,source,main_problem,needed_service,service_category,importance_level,current_solution,satisfaction_level,would_pay,monthly_budget,urgency_level,frequency_of_problem,preferred_solution_type,opinion_text,date_collected
0,1,45-54,Prefer not to say,Ras Al Khaimah rural area,Small business owner,qr_survey,Difficulty receiving online orders,Pharmacy delivery,Delivery,High,Ask family member,Neutral,Yes,51-100 AED,Soon,Weekly,Phone call,We need faster delivery because shops are far ...,2026-04-21
1,2,45-54,Female,Khor Fakkan outskirts,Tourism worker,shop_owner_note,No nearby appliance repair,Local maintenance service,Repair,High,Ask neighbors,Dissatisfied,No,0 AED,Not urgent,Weekly,In-person service,Home maintenance is expensive because provider...,2026-06-23
2,3,45-54,Female,Kalba,Shop owner,shop_owner_note,No regular local bus,Shared taxi service,Transport,High,Ask relatives,Neutral,No,0 AED,Soon,Weekly,Community center,Some people depend on relatives for transport.,2026-04-15
3,4,45-54,Prefer not to say,Khor Fakkan outskirts,Driver,voice_note,Local guides are not visible,Local experience booking,Tourism,Medium,Ask locals,Satisfied,Yes,31-50 AED,Soon,Weekly,Delivery service,Local families could offer farm visits or cult...,2026-05-10
4,5,45-54,Male,Ghayathi,Farmer,paper_form,Limited transport,Shared taxi service,Transport,High,Use taxi,Satisfied,Maybe,100+ AED,Soon,Weekly,WhatsApp,Taxi services are expensive or not always avai...,2026-05-04


In [18]:
import pandas as pd
from pathlib import Path
import re


def get_next_response_number(existing_df):

    if existing_df is None or existing_df.empty:
        return 1

    if "response_id" not in existing_df.columns:
        return len(existing_df) + 1

    numbers = []

    for value in existing_df["response_id"].dropna():
        value = str(value).strip()

        match = re.match(r"R(\d+)$", value)

        if match:
            numbers.append(int(match.group(1)))

    if numbers:
        return max(numbers) + 1

    return len(existing_df) + 1


def assign_short_unique_ids(new_df, start_number):

    new_df = new_df.copy()

    if "batch_id" in new_df.columns:
        new_df = new_df.drop(columns=["batch_id"])

    new_df["response_id"] = [
        f"R{number:06d}"
        for number in range(start_number, start_number + len(new_df))
    ]

    return new_df


def append_to_raw_dataset(new_df, raw_output_path):

    raw_output_path = Path(raw_output_path)
    raw_output_path.parent.mkdir(parents=True, exist_ok=True)

    if raw_output_path.exists():
        old_df = pd.read_csv(raw_output_path)

        if "batch_id" in old_df.columns:
            old_df = old_df.drop(columns=["batch_id"])
    else:
        old_df = pd.DataFrame()

    next_number = get_next_response_number(old_df)

    new_df = assign_short_unique_ids(
        new_df=new_df,
        start_number=next_number
    )

    if not old_df.empty:
        combined_df = pd.concat([old_df, new_df], ignore_index=True)
    else:
        combined_df = new_df.copy()

    before_dedup = len(combined_df)

    combined_df = combined_df.drop_duplicates(
        subset=["response_id"],
        keep="last"
    )

    duplicates_removed = before_dedup - len(combined_df)

    combined_df.to_csv(raw_output_path, index=False)

    print("Raw dataset updated.")
    print("Saved to:", raw_output_path)
    print("New rows added:", len(new_df))
    print("First new response ID:", new_df["response_id"].iloc[0])
    print("Last new response ID:", new_df["response_id"].iloc[-1])
    print("Rows before duplicate removal:", before_dedup)
    print("Duplicates removed:", duplicates_removed)
    print("Final rows:", len(combined_df))

    return combined_df


output_path = PROJECT_ROOT / "data" / "raw" / "responses_raw.csv"

combined_df = append_to_raw_dataset(
    new_df=df,
    raw_output_path=output_path
)

print("Saved CSV to:", output_path)

Raw dataset updated.
Saved to: C:\Users\ASUS\PycharmProjects\PythonProject8\data\raw\responses_raw.csv
New rows added: 250
First new response ID: R000251
Last new response ID: R000500
Rows before duplicate removal: 500
Duplicates removed: 0
Final rows: 500
Saved CSV to: C:\Users\ASUS\PycharmProjects\PythonProject8\data\raw\responses_raw.csv


In [19]:
print("Shape:", df.shape)
print()
print("Columns:")
print(df.columns.tolist())
print()
print("Service categories:")
print(df["service_category"].value_counts())
print()
print("Sources:")
print(df["source"].value_counts())

Shape: (250, 19)

Columns:
['response_id', 'age_group', 'gender', 'region', 'occupation', 'source', 'main_problem', 'needed_service', 'service_category', 'importance_level', 'current_solution', 'satisfaction_level', 'would_pay', 'monthly_budget', 'urgency_level', 'frequency_of_problem', 'preferred_solution_type', 'opinion_text', 'date_collected']

Service categories:
service_category
Transport      48
Tourism        46
Delivery       45
Agriculture    45
Education      39
Repair         27
Name: count, dtype: int64

Sources:
source
paper_form         56
shop_owner_note    51
qr_survey          50
interview          49
voice_note         44
Name: count, dtype: int64


## Next step

Run the cleaning agent:

```powershell
python agents/cleaning_agent.py
```